# COF 膜–水–Na：短程平衡分析

| 文件 | 用途 |
|------|------|
| `log.lammps` | 温度 / $P_{zz}$ / $L_z$ |
| `result_atoms.eq.data` + `result_atoms.lammpstrj` | NPT(z) 段轨迹 → nglview |
| `result_atoms.xyz` + `result_box.dat` + `result_connect.dat` | 由 helper 导出，供 `vmd.tcl` |

- `_helper_functions.py` — `load_lammps_universe` / `read_result_thermo` / `write_xyz` / `save_nglview_frame`

模拟：NVT 100 ps + NPT(z) 400 ps @ 300 K、1 atm（见 `in.lmp`）。


In [ ]:
%reload_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = "retina"
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "figure.facecolor": "white",
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.2,
    "font.size": 10,
})
from MDAnalysis import transformations as trans
import nglview as nv
import warnings
warnings.filterwarnings("ignore")

from _helper_functions import (
    load_lammps_universe,
    read_result_thermo,
    move_origin_to_corner,
    move_origin_to_center,
    save_nglview_frame,
)

# ---- 本例参数 ----
TOPO = "./result_atoms.eq.data"   # NPT 段起点
TRAJ = "./result_atoms.lammpstrj"
DT_FS = 5000.0                    # dump_every * timestep [fs]

# 残基名：与 LAMMPS mol/resid 升序一致（同 C02 写法）
#   resid 1–5 → COF；6–1005 → H2O；1006–1035 → Na
RESNAMES = ["COF"] * 5 + ["H2O"] * 1000 + ["Na"] * 30


## 1. 温度、$P_{zz}$、$L_z$

读全部 thermo 块：seg0 ≈ NVT，seg1 ≈ NPT(z)。列名小写。


In [ ]:
thermo_all = read_result_thermo("log.lammps", segment=None)
print("thermo blocks / rows:", thermo_all["segment"].nunique(), len(thermo_all))
print("columns:", list(thermo_all.columns))

tcol = "temp"

fig, axes = plt.subplots(1, 3, figsize=(10.5, 2.6))
for seg, g in thermo_all.groupby("segment"):
    t_ps = g["time"].to_numpy() / 1000.0  # fs → ps（各段内从 0 计）
    label = f"seg{seg}"
    axes[0].plot(t_ps, g[tcol], lw=1.0, label=label)
    axes[1].plot(t_ps, g["pzz"], lw=1.0, label=label)
    axes[2].plot(t_ps, g["lz"], lw=1.0, label=label)

axes[0].set_ylabel("T [K]")
axes[0].axhline(300.0, color="k", ls="--", lw=0.8)
axes[1].set_ylabel(r"$P_{zz}$ [atm]")
axes[1].axhline(1.0, color="k", ls="--", lw=0.8)
axes[2].set_ylabel(r"$L_z$ [Å]")
for ax in axes:
    ax.set_xlabel("time in segment [ps]")
    ax.tick_params(direction="in")
    ax.legend(fontsize=8, frameon=False)

fig.tight_layout()
fig.savefig("result_thermo_T_Pzz_Lz.png", bbox_inches="tight")
plt.show()

npt = read_result_thermo("log.lammps", segment=-1)
print(
    f"NPT(z) mean: T={npt[tcol].mean():.2f} K, "
    f"Pzz={npt['pzz'].mean():.2f} atm, "
    f"Lz={npt['lz'].mean():.3f} Å"
)


## 2. nglview 可视化

与 [机械控压](../../在线资源/02-实战案例/C02-Lammps机械控压.md) 相同：读 traj 时传入 `resnames=RESNAMES`（按 **LAMMPS mol/resid 升序** 赋名：5×COF、1000×H2O、30×Na）。显示时按 **residue** `wrap`。

**注意：** nglview **不支持动态盒子**（NPT 段 $L_z$ 在变时，画面里的晶胞框不会跟着每帧更新）；这里只做快速验结构，正式看盒与分层出图用 VMD。


In [ ]:
u_view = load_lammps_universe(
    TOPO, TRAJ,
    dt_fs=DT_FS,
    resnames=RESNAMES,
)
u_view.trajectory.add_transformations(
    move_origin_to_corner,
    trans.wrap(u_view.atoms, compound="residues"),
)

view = nv.show_mdanalysis(u_view)
view.clear_representations()
view.add_representation("licorice", selection="COF", radius=0.2)
view.add_representation("ball+stick", selection="H2O", radius=0.12)
view.add_representation("spacefill", selection="Na", radius=0.8)
view.add_unitcell()
view


In [ ]:
# 导出某一帧为 PNG（默认最后一帧）
# save_nglview_frame(view, "result_nglview.png")  # frame=-1 → last


## 3. 导出 XYZ / connect（供 VMD）

`corner → wrap(residues) → center` 后，每隔 10 帧写出 `result_atoms.xyz`、`result_box.dat`，以及 **`connect=True`** 时的 `result_connect.dat`（拓扑 CONECT）。纯 XYZ 无键、VMD 会猜键；带 connect 更准。`vmd.tcl`：COF/Na **VDW 0.5@60**，水 **Licorice 0.2@60**（画键时偶发跨盒长线属正常）。


In [ ]:
u_xyz = load_lammps_universe(TOPO, TRAJ, dt_fs=DT_FS, resnames=RESNAMES)
u_xyz.trajectory.add_transformations(
    move_origin_to_corner,
    trans.wrap(u_xyz.atoms, compound="residues"),
    move_origin_to_center,
)
n = u_xyz.trajectory.n_frames
print(f"frames = {n}")

# 每隔 10 帧 → xyz + box + connect（供 vmd.tcl）
XYZ_STEP = 10
u_xyz.write_xyz(connect=True, start=0, stop=n, step=XYZ_STEP, dir=".")
print(
    f"wrote: result_atoms.xyz, result_box.dat, result_connect.dat  "
    f"(every {XYZ_STEP} frames, ~{(n + XYZ_STEP - 1) // XYZ_STEP} frames)"
)

## 4. 结果汇总


In [ ]:
summary = pd.DataFrame(
    [
        {"quantity": "T", "value": npt[tcol].mean(), "unit": "K"},
        {"quantity": "Pzz", "value": npt["pzz"].mean(), "unit": "atm"},
        {"quantity": "Lz", "value": npt["lz"].mean(), "unit": "Å"},
        {"quantity": "Lx", "value": npt["lx"].mean(), "unit": "Å"},
        {"quantity": "Ly", "value": npt["ly"].mean(), "unit": "Å"},
    ]
)
summary.to_csv("result_summary_eq.csv", index=False)
summary
